In [18]:
import pandas as pd

df = pd.read_csv("emails_raw.csv")

# 1. Check data file
print(df.head()) #xem vài dòng đầu của dataset
print(df.shape)
print(df.columns)
print(df.info())

                                                text  spam
0  Subject: naturally irresistible your corporate...     1
1  Subject: the stock trading gunslinger  fanny i...     1
2  Subject: unbelievable new homes made easy  im ...     1
3  Subject: 4 color printing special  request add...     1
4  Subject: do not have money , get software cds ...     1
(5728, 2)
Index(['text', 'spam'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    5728 non-null   str  
 1   spam    5728 non-null   int64
dtypes: int64(1), str(1)
memory usage: 89.6 KB
None


In [17]:
# 2. Remove missing data & duplicate emails
print(df.isnull().sum())
# if df.isnull().values.any():
#     df = df.dropna() #dùng để xoá dòng bị thiếu data

# # Check duplicate emails
# print("Shape before:", df.shape)

# print("Duplicate rows:", df.duplicated().sum())

# df = df.drop_duplicates()

# print("Shape after removing duplicates:", df.shape)

text    0
spam    0
dtype: int64


In [8]:
# 3. Check label có cân bằng không (vì nếu quá lệch nhau => accuracy bị ảnh hưởng)
# Độ lệch 70:30 đến 85:15 là đẹp
print(df["spam"].value_counts())

spam
0    4327
1    1368
Name: count, dtype: int64


In [9]:
# 4. Text cleaning
import re

def clean(text):
    # Chuyển toàn bộ thành chữ thường
    text = text.lower()
    # Remove email prefixes
    text = re.sub(
        r'((re|fw|fwd)\s*:)+',
        ' ',
        text
    )

    # remove "subject:"
    text = re.sub(
        r'subject\s*:',
        ' ',
        text
    )
    # Xoá URL
    text = re.sub(r'http\S+|www\S+',' ',text)
    # Xóa HTML tags
    text = re.sub(r'<.*?>',' ',text)
    # Xoá mọi thứ KHÔNG phải chữ cái
    text = re.sub(r'[^a-zA-Z ]',' ',text)
    # Dọn khoảng trắng thừa
    text = re.sub(r'\s+',' ',text).strip()

    return text

df["text"] = df["text"].apply(clean)

df.to_csv(
    "emails_cleaned.csv",
    index=False
)

In [10]:
# 5. Chuyển text → số (Vì ML ko hiểu chữ)
from sklearn.feature_extraction.text import TfidfVectorizer

# X = input features (text)
X = df["text"]

# y = label
y = df["spam"]

# Create TF-IDF converter
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    min_df=2, # giữ những từ xuất hiện ở ít nhất 2 email.
    ngram_range=(1,2) # giữ ngữ cảnh cho từ vựng
)

# Convert text -> numbers
X = vectorizer.fit_transform(X)

print(X.shape)

(5695, 10000)


In [12]:
# 6. Lưu vectorixer
import joblib
joblib.dump(
    vectorizer,
    "tfidf_vectorizer.pkl"
)
print("Vectorizer saved!")

# lưu feature count để debug
print(
    "Vocabulary size:",
    len(vectorizer.vocabulary_)
)

Vectorizer saved!
Vocabulary size: 10000


In [14]:
# 7. Chia train / test
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y #công dụng: Chia train/test nhưng phải giữ nguyên tỷ lệ các lớp trong y.
)

# lưu
import joblib

joblib.dump(X_train, "X_train.pkl")
joblib.dump(X_test, "X_test.pkl")

joblib.dump(y_train, "y_train.pkl")
joblib.dump(y_test, "y_test.pkl")

['y_test.pkl']